# Árvore de Decisão — T1 IA

Classifica o estado de um tabuleiro 3x3 do jogo da velha em uma das 4 classes:
- 0: tem_jogo
- 1: x_venceu
- 2: o_venceu
- 3: empate

Encoding: X=1, O=-1, vazio=0.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

ModuleNotFoundError: No module named 'pandas'

## 1. Carregar os dados

Os CSVs já foram divididos fisicamente em treino/validação/teste pelo `main.py` (split estratificado 70/15/15).

In [ ]:
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

X_train = train_df.iloc[:, :9]
y_train = train_df['class']
X_val = val_df.iloc[:, :9]
y_val = val_df['class']
X_test = test_df.iloc[:, :9]
y_test = test_df['class']

print('Treino:    ', len(X_train), 'amostras')
print('Validação: ', len(X_val), 'amostras')
print('Teste:     ', len(X_test), 'amostras')
print('\nDistribuição treino:', y_train.value_counts().sort_index().to_dict())

## 2. Modelo baseline (defaults)

Treina uma árvore de decisão com hiperparâmetros padrão pra ter um ponto de referência.

In [ ]:
baseline = DecisionTreeClassifier(random_state=42)
baseline.fit(X_train, y_train)

y_val_pred = baseline.predict(X_val)
print('Validação (baseline):')
print(f"  Acurácia: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"  F1 macro: {f1_score(y_val, y_val_pred, average='macro'):.4f}")

## 3. Tuning de hiperparâmetros (GridSearchCV)

Os hiperparâmetros mais importantes da árvore:
- `criterion`: critério de divisão (gini ou entropy)
- `max_depth`: profundidade máxima (controla overfitting)
- `min_samples_split`: mínimo de amostras pra dividir um nó
- `min_samples_leaf`: mínimo de amostras numa folha
- `class_weight`: 'balanced' compensa o desbalanceamento da classe `empate` (32 amostras vs 200 das outras)

Métrica do tuning: **F1 macro** (em vez de acurácia), porque a acurácia é enganosa com classes desbalanceadas.

In [ ]:
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 4, 6, 8, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'class_weight': [None, 'balanced'],
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_train_val, y_train_val)

print('Melhores parâmetros:', grid.best_params_)
print(f"Melhor F1 macro (CV): {grid.best_score_:.4f}")

## 4. Avaliação no conjunto de teste

Métricas exigidas pelo enunciado: **acurácia, precision, recall, F-measure (F1)**.

In [ ]:
best = grid.best_estimator_
y_pred = best.predict(X_test)

print(f"Acurácia:        {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision macro: {precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}")
print(f"Recall macro:    {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 macro:        {f1_score(y_test, y_pred, average='macro'):.4f}")

print('\nRelatório por classe:')
nomes = ['tem_jogo', 'x_venceu', 'o_venceu', 'empate']
print(classification_report(y_test, y_pred, target_names=nomes, zero_division=0))

## 5. Matriz de confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=nomes)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Matriz de confusão — Árvore de Decisão')
plt.tight_layout()
plt.show()

## 6. Visualização da árvore

Útil pro relatório PPT: mostra como a árvore está tomando as decisões.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    best,
    feature_names=[f'pos_{i}' for i in range(9)],
    class_names=nomes,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
)
plt.tight_layout()
plt.show()

## 7. Importância das features

Mostra quais posições do tabuleiro a árvore considera mais relevantes pra classificar.

In [ ]:
importancias = pd.Series(
    best.feature_importances_,
    index=[f'pos_{i}' for i in range(9)],
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importancias.plot(kind='barh', ax=ax)
ax.set_title('Importância das features (posições do tabuleiro)')
ax.set_xlabel('Importância')
plt.tight_layout()
plt.show()

## 8. Salvar o modelo

In [ ]:
joblib.dump(best, 'decision_tree_model.pkl')
print('Modelo salvo em decision_tree_model.pkl')